In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Step 1: Load the data (change path if needed)
dataset_path = Path(r"/content/Dataset .csv")
dataset_df = pd.read_csv(dataset_path)
# Target column we want to predict
target_rating = "Aggregate rating"

# These two look like they directly tell you the rating, so we'll drop them
excluded_columns = ["Rating color", "Rating text"]

feature_data = dataset_df.drop(columns=[target_rating] + excluded_columns)
target_data = dataset_df[target_rating]


# Step 2: Split into train and test sets (80/20)
from sklearn.model_selection import train_test_split

train_data, test_data, train_target, test_target = train_test_split(
    feature_data, target_data, test_size=0.2, random_state=42
)


# Step 3: Set up preprocessing pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Separate columns into numeric and categorical (some bools might sneak in)
numeric_columns = train_data.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = train_data.select_dtypes(exclude=["number", "bool"]).columns.tolist()

# Probably safe to fill missing numbers with median
numeric_pipeline = Pipeline(steps=[
    ("fill_missing", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

# Fill missing categories with mode (most common value), then one-hot encode
categorical_pipeline = Pipeline(steps=[
    ("fill_missing", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing for all columns
data_processor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_columns),
    ("cat", categorical_pipeline, categorical_columns)
])


# Step 4: Train Gradient Boosting model
from sklearn.ensemble import GradientBoostingRegressor

# Using basic GradientBoosting — haven’t tuned params yet
gradient_model = GradientBoostingRegressor(random_state=42)

# Full pipeline: preprocess → model
model_pipeline = Pipeline(steps=[
    ("preprocessing", data_processor),
    ("regressor", gradient_model)
])

# Fit everything on training data
model_pipeline.fit(train_data, train_target)


# Step 5: Evaluate model on the test set
from sklearn.metrics import mean_squared_error, r2_score

predicted_values = model_pipeline.predict(test_data)

rmse_score = np.sqrt(mean_squared_error(test_target, predicted_values))
r2_score_value = r2_score(test_target, predicted_values)

print("\nModel Performance on Test Set:")
print(f" - RMSE: {rmse_score:.3f}")
print(f" - R² Score: {r2_score_value:.3f}")


# Step 6: Interpret using permutation importance
from sklearn.inspection import permutation_importance

# Note: This will take a bit of time
print("\nRunning permutation importance... (could be slow)")

importance_result = permutation_importance(
    model_pipeline,
    test_data,
    test_target,
    n_repeats=10,
    random_state=42,
    scoring="neg_root_mean_squared_error"
)

# Match importance to the raw column names
importance_scores = pd.Series(
    importance_result.importances_mean,
    index=test_data.columns
)

top_ranked_features = importance_scores.sort_values(ascending=False).head(15)

print("\nTop 15 Most Influential Features:")
print(top_ranked_features)


Model Performance on Test Set:
 - RMSE: 0.275
 - R² Score: 0.967

Running permutation importance... (could be slow)

Top 15 Most Influential Features:
Votes                   1.865547
Restaurant ID           0.086416
Latitude                0.005574
Restaurant Name         0.004521
Average Cost for two    0.003914
City                    0.003597
Country Code            0.003453
Has Online delivery     0.002793
Currency                0.002695
Cuisines                0.002560
Longitude               0.002049
Price range             0.001550
Locality Verbose        0.000895
Locality                0.000354
Address                 0.000000
dtype: float64


In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, csr_matrix

In [3]:
df_recommendations = pd.read_csv('/content/Dataset.csv')
df_recommendations.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [4]:
# check the rows and columns
df_recommendations.shape

(9551, 21)

In [5]:
df_recommendations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  Switch to order menu 

In [8]:
# Fill missing values in 'Cuisines' column with the mode of the column
df_recommendations['Cuisines'] = df_recommendations['Cuisines'].fillna(df_recommendations['Cuisines'].mode()[0])

TF-IDF Implementation for Cuisines

In [9]:
# Clean restaurant names
df_recommendations['Restaurant Name'] = df_recommendations['Restaurant Name'].str.replace(r'[^\w\s&\'-]', '', regex=True)
df_recommendations.duplicated().sum()
df_recommendations.columns
# Drop unnecessary columns
df_recommendations = df_recommendations.loc[:, ['Restaurant Name','Average Cost for two', 'Cuisines','Aggregate rating']].copy()
# Convert 'Cuisines' to lowercase and remove extra spaces
df_recommendations['Cuisines'] = df_recommendations['Cuisines'].str.lower().str.replace(r'\s*,\s*', ', ', regex=True)
df_recommendations['Cuisines']

,Cuisines
0,"french, japanese, desserts"
1,japanese
2,"seafood, asian, filipino, indian"
3,"japanese, sushi"
4,"japanese, korean"
...,...
9546,turkish
9547,"world cuisine, patisserie, cafe"
9548,"italian, world cuisine"
9549,restaurant cafe


In [10]:
# initialize TfidfVectorizer
tfidf = TfidfVectorizer()
# Fit and transform the 'Cuisines' column to create a TF-IDF matrix
tfidf_matrix = tfidf.fit_transform(df_recommendations['Cuisines'])
# Display the shape of the TF-IDF matrix
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 27090 stored elements and shape (9551, 150)>

In [13]:
scaler = MinMaxScaler()
cost_scaled = scaler.fit_transform(df_recommendations[['Average Cost for two']])
# Convert the scaled cost to sparse matrix format
cost_sparse = csr_matrix(cost_scaled)
# Combine the TF-IDF matrix and the scaled cost matrix
combined_features = hstack([tfidf_matrix, cost_sparse])
# Calculate cosine similarity b/w the combined features
cos_sim = cosine_similarity(combined_features,combined_features)

In [14]:
# Function to get restaurant recommendations based on user preferences
def filter_by_preferences(df, cuisine, budget, rating):
    #Filter restaurants based on user preferences
    return df[
        df['Cuisines'].str.contains(cuisine.lower(), case=False, na=False) &
        (df['Average Cost for two'] <= budget) &
        (df['Aggregate rating'] >= rating)
    ]
    # Function to recommend similar restaurants based on user preferences
def recommend_similar(df, similarity_matrix, filter_df, budget, rating, tops=5):

    if filter_df.empty:
        print("No restaurants found matching your criteria.")
        return pd.DataFrame()
    # Use the best-rated restaurant from filtered results
    best_restaurant_idx = filter_df['Aggregate rating'].idxmax()
    # Get similarity scores for the best restaurant
    sim_score = list(enumerate(similarity_matrix[best_restaurant_idx]))
    # Sort by similarity score (descending) and exclude the restaurant itself
    sim_score = sorted(sim_score,key=lambda x: x[1], reverse=True)[1:]

# Get recommendations based on budget and rating constraints
    # Initialize an empty list to store recommendations
    recommendations = []
    # Iterate through the sorted similarity scores
    for i, score in sim_score:
        # Get the restaurant at the current index
        restaurant = df.iloc[i]
        # check if the restaurant meets budget and rating criteria and if we haven't reached the limit of recommendations
        if (restaurant['Average Cost for two'] <= budget and
            restaurant['Aggregate rating'] >= rating and
            len(recommendations) < tops):
            recommendations.append(i)
    return df.iloc[recommendations]

In [15]:
#### Hybrid Recommender Function

# This function combines user preferences filtering and similarity-based recommendations
def hybrid_recommender(df, cuisine, budget, rating, top_n=5):
    # First filter by user preferences
    filtered_df = filter_by_preferences(df, cuisine, budget, rating)

    if filtered_df.empty:
        print(f"No restaurants found for cuisine: {cuisine}, budget: {budget}, rating: {rating}")
        return pd.DataFrame()

    print(f"Restaurants found for cuisine: {cuisine}, budget: {budget}, rating: {rating}")
    print(f"Filtered matches: {len(filtered_df)} | Top N requested: {top_n}")


    # Option 1: Return top restaurants from filtered results (simpler and more reliable)
    if len(filtered_df) <= top_n:
        print("Used Option 1: returning top rated filtered matches (no similarity).")
        return filtered_df.sort_values('Aggregate rating', ascending=False)

    # Option 2: Use similarity-based recommendations
    print("Used Option 2: similarity based recommendations activated.")
    recommendations = recommend_similar(df, cos_sim, filtered_df, budget, rating, top_n)

    return recommendations

In [16]:
# function to show recommendations
def show_recommendations(df):
    if df.empty:
        print("No recommendations found.")
        return
    else:
        return df[['Restaurant Name', 'Cuisines', 'Average Cost for two', 'Aggregate rating']].reset_index(drop=True)

In [17]:
show_recommendations(hybrid_recommender(df_recommendations,cuisine="Lucknowi",budget=1000,rating=4.0,top_n=5))

Restaurants found for cuisine: Lucknowi, budget: 1000, rating: 4.0
Filtered matches: 1 | Top N requested: 5
Used Option 1: returning top rated filtered matches (no similarity).


,Restaurant Name,Cuisines,Average Cost for two,Aggregate rating
0,Grandson of Tunday Kababi,"mughlai, lucknowi",300,4.9


In [18]:
show_recommendations(hybrid_recommender(df_recommendations, cuisine="Martian Fusion", budget=50, rating=5))

No restaurants found for cuisine: Martian Fusion, budget: 50, rating: 5
No recommendations found.


In [20]:
from IPython.display import Markdown

Markdown("""
| Test Scenario              | Restaurants Found | System Response          | Success Rate     |
|---------------------------|-------------------|---------------------------|------------------|
| Japanese (₹1200, ≥4.5)    | 11 matches        | Content-based similarity | ✅ Excellent     |
| Mexican (₹700, ≥4.0)      | 40 matches        | Content-based similarity | ✅ Excellent     |
| Lucknowi (₹1200, ≥4.0)    | 1 match           | Direct recommendation    | ✅ Good          |
| Martian Fusion (₹50, 5.0) | 0 matches         | Clear error message      | ✅ Appropriate   |
""")


| Test Scenario              | Restaurants Found | System Response          | Success Rate     |
|---------------------------|-------------------|---------------------------|------------------|
| Japanese (₹1200, ≥4.5)    | 11 matches        | Content-based similarity | ✅ Excellent     |
| Mexican (₹700, ≥4.0)      | 40 matches        | Content-based similarity | ✅ Excellent     |
| Lucknowi (₹1200, ≥4.0)    | 1 match           | Direct recommendation    | ✅ Good          |
| Martian Fusion (₹50, 5.0) | 0 matches         | Clear error message      | ✅ Appropriate   |
